In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from pathlib import Path

# Load .env from the root folder
BASE_DIR = Path.cwd().parent
load_dotenv(dotenv_path=BASE_DIR / ".env")

# Connect to Neon
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

def run_query(sql_query):
    """Execute a SQL query and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(sql_query, conn)

print("✅ Environment ready. Connected to Neon.")

✅ Environment ready. Connected to Neon.


In [3]:
# RFM Analysis
query_rfm = """
WITH customer_metrics AS (
    SELECT 
        "CustomerID",
        MAX("InvoiceDate")::date AS last_purchase_date,
        COUNT(DISTINCT "Invoice") AS frequency,
        SUM("TotalAmount") AS monetary
    FROM retail_transactions
    GROUP BY "CustomerID"
),
rfm AS (
    SELECT 
        "CustomerID",
        (SELECT MAX("InvoiceDate")::date FROM retail_transactions) - last_purchase_date AS recency_days,
        frequency,
        monetary
    FROM customer_metrics
),
rfm_scores AS (
    SELECT 
        "CustomerID",
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm
)
SELECT 
    "CustomerID",
    recency_days,
    frequency,
    ROUND(monetary::numeric, 2) AS monetary,
    r_score, f_score, m_score,
    CASE 
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champion'
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyal'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customer'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
        ELSE 'Potential Loyalist'
    END AS segment
FROM rfm_scores;
"""

df_rfm = run_query(query_rfm)
print(f"Total customers segmented: {len(df_rfm)}")
print("\nSegment Distribution:")
print(df_rfm['segment'].value_counts())
print("\nRevenue by Segment:")
print(df_rfm.groupby('segment')['monetary'].agg(['count', 'sum', 'mean']).round(2).sort_values('sum', ascending=False))

Total customers segmented: 5878

Segment Distribution:
segment
Lost                  1519
Champion              1342
Loyal                 1095
At Risk                833
Potential Loyalist     641
New Customer           448
Name: count, dtype: int64

Revenue by Segment:
                    count          sum     mean
segment                                        
Champion             1342  11947714.02  8902.92
Loyal                1095   2485219.57  2269.61
At Risk               833   1615045.81  1938.83
Lost                 1519    631290.25   415.60
New Customer          448    388133.35   866.37
Potential Loyalist    641    307401.25   479.57


In [4]:
fig = px.scatter(
    df_rfm.sample(min(2000, len(df_rfm))),  # Sample for performance
    x='frequency', 
    y='monetary',
    size='monetary',
    color='segment',
    hover_data=['CustomerID', 'recency_days'],
    title='🎯 RFM Customer Segmentation',
    labels={'frequency': 'Number of Orders', 'monetary': 'Total Spent (£)', 'segment': 'Segment'}
)
fig.update_layout(height=600)
fig.show()

In [6]:
print("Shape:", df_pareto.shape)
print("Max cumulative_pct:", df_pareto['cumulative_pct'].max())
print(df_pareto.head())

Shape: (500, 6)
Max cumulative_pct: 60.6
   rnk StockCode                         Description    revenue  \
0    1     22423            REGENCY CAKESTAND 3 TIER  277656.25   
1    2    85123A  WHITE HANGING HEART T-LIGHT HOLDER  247048.01   
2    3     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60   
3    4         M                              Manual  151777.67   
4    5    85099B             JUMBO BAG RED RETROSPOT  134307.44   

   cumulative_pct  product_pct  
0            1.60         0.02  
1            3.02         0.04  
2            3.99         0.06  
3            4.86         0.08  
4            5.64         0.09  


In [7]:
query_pareto = """
WITH product_revenue AS (
    SELECT 
        "StockCode",
        "Description",
        SUM("TotalAmount") AS revenue
    FROM retail_transactions
    GROUP BY "StockCode", "Description"
    HAVING SUM("TotalAmount") > 0
),
ranked AS (
    SELECT 
        "StockCode",
        "Description",
        revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rnk,
        COUNT(*) OVER () AS total_products
    FROM product_revenue
)
SELECT 
    rnk,
    "StockCode",
    "Description",
    ROUND(revenue::numeric, 2) AS revenue,
    ROUND((cumulative_revenue / total_revenue * 100)::numeric, 2) AS cumulative_pct,
    ROUND((rnk::numeric / total_products * 100), 2) AS product_pct
FROM ranked
ORDER BY rnk;
"""

df_pareto = run_query(query_pareto)
print(f"Total products in dataset: {len(df_pareto)}")
print(f"Max cumulative %: {df_pareto['cumulative_pct'].max()}%")

Total products in dataset: 5315
Max cumulative %: 100.0%


In [8]:
# Find where cumulative revenue reaches 80%
pareto_80 = df_pareto[df_pareto['cumulative_pct'] >= 80]

if len(pareto_80) > 0:
    row = pareto_80.iloc[0]
    print("📊 Pareto Analysis Result:")
    print(f"   Total products in dataset: {len(df_pareto)}")
    print(f"   Products generating 80% revenue: {int(row['rnk'])}")
    print(f"   % of products generating 80% revenue: {row['product_pct']}%")
    print(f"   💡 Insight: {row['product_pct']}% of products drive 80% of revenue.")
else:
    print("⚠️ Even 100% of products don't reach 80% — check data!")

📊 Pareto Analysis Result:
   Total products in dataset: 5315
   Products generating 80% revenue: 1138
   % of products generating 80% revenue: 21.41%
   💡 Insight: 21.41% of products drive 80% of revenue.


In [9]:
fig = px.line(
    df_pareto,
    x='rnk',
    y='cumulative_pct',
    title='📊 Pareto Curve: Product Revenue Concentration',
    labels={'rnk': 'Product Rank (by revenue)', 'cumulative_pct': 'Cumulative Revenue %'}
)
fig.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80% threshold")
fig.add_vline(x=pareto_80.iloc[0]['rnk'], line_dash="dash", line_color="green")
fig.show()

In [10]:
query_growth = """
WITH monthly_revenue AS (
    SELECT 
        DATE_TRUNC('month', "InvoiceDate")::date AS month,
        SUM("TotalAmount") AS revenue,
        COUNT(DISTINCT "Invoice") AS orders,
        COUNT(DISTINCT "CustomerID") AS customers
    FROM retail_transactions
    GROUP BY DATE_TRUNC('month', "InvoiceDate")
)
SELECT 
    month,
    ROUND(revenue::numeric, 2) AS revenue,
    orders,
    customers,
    ROUND((revenue / orders)::numeric, 2) AS aov,
    ROUND(LAG(revenue) OVER (ORDER BY month)::numeric, 2) AS prev_month_revenue,
    ROUND(((revenue - LAG(revenue) OVER (ORDER BY month)) / 
           NULLIF(LAG(revenue) OVER (ORDER BY month), 0) * 100)::numeric, 2) AS mom_growth_pct
FROM monthly_revenue
ORDER BY month;
"""

df_growth = run_query(query_growth)
print(df_growth.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Bar(x=df_growth['month'], y=df_growth['revenue'], name='Revenue', marker_color='#4CAF50'))
fig.add_trace(go.Scatter(x=df_growth['month'], y=df_growth['mom_growth_pct'], name='MoM Growth %', 
                          yaxis='y2', line=dict(color='#FF9800', width=2)))
fig.update_layout(
    title='📊 Monthly Revenue & Growth %',
    yaxis=dict(title='Revenue (£)'),
    yaxis2=dict(title='MoM Growth %', overlaying='y', side='right'),
    height=500
)
fig.show()

     month    revenue  orders  customers    aov  prev_month_revenue  mom_growth_pct
2009-12-01  683504.01    1512        955 452.05                 NaN             NaN
2010-01-01  555802.67    1011        720 549.76           683504.01          -18.68
2010-02-01  504558.96    1104        772 457.03           555802.67           -9.22
2010-03-01  696978.47    1524       1057 457.33           504558.96           38.14
2010-04-01  591982.00    1329        942 445.43           696978.47          -15.06
2010-05-01  597833.38    1377        966 434.16           591982.00            0.99
2010-06-01  636371.13    1497       1041 425.10           597833.38            6.45
2010-07-01  589736.17    1381        928 427.04           636371.13           -7.33
2010-08-01  602224.60    1293        911 465.76           589736.17            2.12
2010-09-01  829013.95    1689       1145 490.83           602224.60           37.66
2010-10-01 1033112.01    2133       1497 484.35           829013.95         

In [11]:
query_country_perf = """
SELECT 
    "Country",
    COUNT(DISTINCT "CustomerID") AS customers,
    COUNT(DISTINCT "Invoice") AS orders,
    ROUND(SUM("TotalAmount")::numeric, 2) AS revenue,
    ROUND((SUM("TotalAmount") / COUNT(DISTINCT "Invoice"))::numeric, 2) AS aov
FROM retail_transactions
GROUP BY "Country"
HAVING COUNT(DISTINCT "CustomerID") >= 20
ORDER BY revenue DESC
LIMIT 15;
"""

df_country_perf = run_query(query_country_perf)
print(df_country_perf.to_string(index=False))

df_intl = df_country_perf[df_country_perf['Country'] != 'United Kingdom'].head(10)

if len(df_intl) > 0:
    fig = px.scatter(
        df_intl,
        x='customers',
        y='aov',
        size='revenue',
        color='Country',
        title='🌍 International Markets: AOV vs Customer Count',
        labels={'customers': 'Unique Customers', 'aov': 'Avg Order Value (£)'}
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("⚠️ No international markets with ≥20 customers found.")

       Country  customers  orders     revenue     aov
United Kingdom       5350   33541 14389234.92  429.00
   Netherlands         22     228   554038.09 2429.99
       Germany        107     789   425019.71  538.68
        France         95     614   348768.96  568.03
         Spain         41     154   108332.49  703.46
   Switzerland         22      90   100061.94 1111.80
       Belgium         29     149    65387.82  438.84
      Portugal         24      93    55554.78  597.36
